In [4]:
#import
import pandas as pd

In [5]:
# Charger ton fichier Excel
df = pd.read_excel("../fichiers/reseau_en_arbre.xlsx")  

On regroupe par infrastructure, on veut connaître :

le nombre de bâtiments connectés à chaque infra_id,

la somme totale de maisons qu’elle dessert.

In [6]:
infra_stats = df.groupby("infra_id").agg(
    n_bat_servis=("id_batiment","nunique"),
    total_maisons=("nb_maisons","sum"),
    infra_type=("infra_type","first"),
    longueur=("longueur","first")
)

On calcule la difficulté d’infrastructure :

plus c’est long et moins il y a de bâtiments, plus c’est difficile.

In [7]:
infra_stats["diff_infra"] = infra_stats["longueur"] / infra_stats["n_bat_servis"]


On attribue une priorité d’infrastructure

On considère :

intacte = priorité 0

à_remplacer + diff_infra ≤ 5 → priorité 1

à_remplacer + diff_infra entre 5 et 20 → priorité 2

à_remplacer + diff_infra > 20 → priorité 3

In [8]:
def prio(row):
    if row.infra_type == "infra_intacte": return 0
    if row.diff_infra <= 5: return 1
    if row.diff_infra <= 20: return 2
    return 3

infra_stats["priorite"] = infra_stats.apply(prio, axis=1)

On recalcule la difficulté de chaque bâtiment

En sommant les difficultés des infras qui le desservent :

In [10]:
infra_stats = infra_stats.reset_index()

bat_stats = df.merge(infra_stats[["infra_id","diff_infra"]], on="infra_id", how="left")
bat_diffs = bat_stats.groupby("id_batiment").agg(
    diff_batiment=("diff_infra","sum"),
    nb_maisons=("nb_maisons","first")
).reset_index()

bat_diffs["score"] = bat_diffs["diff_batiment"] / bat_diffs["nb_maisons"]

On classe les bâtiments par difficulté croissante (et par nombre de maisons décroissant)

Cela donne l’ordre de priorisation :

In [11]:
bat_diffs = bat_diffs.sort_values(by="score").reset_index(drop=True)
bat_diffs["rang"] = bat_diffs.index + 1


In [12]:
# Export Excel
infra_stats.to_excel("priorisation_infra.xlsx", index=False)
bat_diffs.to_excel("priorisation_batiment.xlsx", index=False)

print("✅ Fichiers exportés : priorisation_infra.xlsx et priorisation_batiment.xlsx")

✅ Fichiers exportés : priorisation_infra.xlsx et priorisation_batiment.xlsx
